# 10 — Modelo em duas etapas para atraso de contratos

Este notebook transforma a ideia do notebook 09 em um **modelo hurdle / duas etapas**:

1. **Etapa 1 — classificação:** estima a probabilidade de o contrato atrasar.
2. **Etapa 2 — severidade:** entre contratos que atrasaram, estima o atraso relativo ao prazo ajustado.
3. **Previsão final em dias:**

`probabilidade_de_atraso × atraso_relativo_condicional × prazo_ajustado`

A vantagem é separar duas perguntas diferentes:

- **vai atrasar?**
- **se atrasar, quanto?**

Isso é adequado para uma base em que muitos contratos têm `target_dias_atraso = 0`.

A validação continua temporal: 4 janelas expanding nos 40 contratos mais antigos e teste final nos 8 mais recentes.

## O que foi mantido e o que mudou em relação ao notebook 09

**Mantido:** construção do target, joins, features por contrato, dois cenários (ordem de serviço e execução), poda de redundância, separação temporal e teste final reservado.

**Removido:** regressão única tentando explicar ao mesmo tempo zeros e atrasos positivos.

**Novo:** classificação + regressão condicional aos contratos atrasados. A previsão final é contínua em dias e não depende de um threshold arbitrário.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

N_TEST = 8
MIN_TRAIN = 20
BLOCK = 5
N_FOLDS = 4

LIMITE_CORRELACAO = 0.85

# Se necessário, defina manualmente a pasta dos CSVs.
DATA_DIR = None

## 1. Carga dos dados

In [ ]:
    FILES = {
    "contratos": "vw_modelagem_contratos_filtros.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "parcelas": "vw_bi_parcelas.csv",
}


def find_data_dir():
    if DATA_DIR is not None:
        return Path(DATA_DIR)

    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd / "data",
        cwd / "data" / "raw",
        Path("/mnt/user-data/uploads"),
    ]

    for parent in cwd.parents:
        candidates += [
            parent / "data",
            parent / "data" / "raw",
        ]

    for folder in candidates:
        if (folder / FILES["contratos"]).exists():
            return folder

    raise FileNotFoundError(
        "CSVs não encontrados. Defina DATA_DIR."
    )


base = find_data_dir()


def read(name, **kwargs):
    return pd.read_csv(
        base / FILES[name],
        encoding="utf-8-sig",
        **kwargs,
    )


contratos = read(
    "contratos",
    parse_dates=[
        "data_assinatura",
        "data_ordem_servico",
        "data_recebimento_provisorio",
    ],
)

aditivos_prazo = read(
    "aditivos_prazo",
    parse_dates=[
        "Data_termino_anterior",
        "Data_termino_nova",
    ],
)

aditivos_valor = read("aditivos_valor")
liberacoes = read("liberacoes")
parcelas = read("parcelas")

assert contratos["idContrato"].is_unique

ids = set(contratos["idContrato"])

aditivos_prazo = aditivos_prazo[
    aditivos_prazo["idContrato"].isin(ids)
].copy()

aditivos_valor = aditivos_valor[
    aditivos_valor["idContrato"].isin(ids)
].copy()

liberacoes = liberacoes[
    liberacoes["id_Contrato"].isin(ids)
].copy()

parcelas = parcelas[
    parcelas["id_contrato"].isin(ids)
].copy()

print("Pasta:", base)
print("Contratos:", len(contratos))

Pasta: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw
Contratos: 48


## 2. Construção do target

O atraso é calculado contra o prazo ajustado pelos aditivos de prazo.

O notebook mantém dois targets:

- `target_atraso`: 0/1 para a etapa de classificação;
- `target_atraso_rel`: intensidade do atraso em relação ao prazo ajustado, usada apenas entre contratos atrasados.

In [3]:
df = contratos.copy()

df["fim_original"] = (
    df["data_ordem_servico"]
    + pd.to_timedelta(
        df["prazo_total"],
        unit="D",
    )
)

fim_aditado = (
    aditivos_prazo
    .groupby("idContrato")["Data_termino_nova"]
    .max()
    .rename("fim_aditado")
)

df = df.merge(
    fim_aditado,
    on="idContrato",
    how="left",
)

df["fim_ajustado"] = df[
    ["fim_original", "fim_aditado"]
].max(axis=1)

df["prazo_ajustado"] = (
    df["fim_ajustado"]
    - df["data_ordem_servico"]
).dt.days

df["duracao_real"] = (
    df["data_recebimento_provisorio"]
    - df["data_ordem_servico"]
).dt.days

df["target_dias_atraso"] = (
    df["duracao_real"]
    - df["prazo_ajustado"]
).clip(lower=0)

df["target_atraso_rel"] = (
    df["target_dias_atraso"]
    / df["prazo_ajustado"]
)

df["target_atraso"] = (
    df["target_dias_atraso"] > 0
).astype(int)

assert (df["prazo_ajustado"] > 0).all()

print("Contratos:", len(df))
print("Com atraso:", int(df["target_atraso"].sum()))
print("Sem atraso:", int((df["target_atraso"] == 0).sum()))

display(
    df[
        [
            "target_dias_atraso",
            "target_atraso_rel",
            "prazo_ajustado",
        ]
    ].describe().T
)

Contratos: 48
Com atraso: 17
Sem atraso: 31


,count,mean,std,min,25%,50%,75%,max
target_dias_atraso,48.000,31.104,73.405,0.000,0.000,0.000,25.750,380.000
target_atraso_rel,48.000,0.239,0.941,0.000,0.000,0.000,0.114,6.333
prazo_ajustado,48.000,311.812,262.816,60.000,131.250,200.500,397.000,"1,110.000"


## 3. Features por contrato

In [4]:
d = (
    df
    .sort_values(
        ["data_ordem_servico", "idContrato"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Informações conhecidas na ordem de serviço
# ------------------------------------------------------------

d["log_valor_total"] = np.log(
    d["valor_total_contrato"]
)

d["log_valor_por_dia"] = np.log(
    d["valor_total_contrato"]
    / d["prazo_total"]
)

d["perc_desconto"] = d["Perc_desconto"]

d["dias_assinatura_ate_os"] = (
    d["data_ordem_servico"]
    - d["data_assinatura"]
).dt.days

# Histórico simples: somente contratos anteriores na ordem temporal.
d["fornecedor_contratos_anteriores"] = (
    d.groupby("id_fornecedor").cumcount()
)

d["predio_contratos_anteriores"] = (
    d.groupby("id_predio").cumcount()
)

# ------------------------------------------------------------
# Parcelas
# ------------------------------------------------------------

p = parcelas.drop_duplicates("id_parcela")

TIPOS = {
    "orig": ["Serviços Originais"],
    "aditivo": ["Aditivo Mercado", "Aditivo Histórico"],
    "supressao": ["Supressão Histórico", "Supressão Mercado"],
    "reajuste": ["Reajuste"],
}

for nome, tipos in TIPOS.items():

    g = (
        p[p["tipo_de_parcela"].isin(tipos)]
        .groupby("id_contrato")
        .agg(
            **{
                f"n_{nome}": (
                    "id_parcela",
                    "nunique",
                ),
                f"valor_{nome}": (
                    "valor_reajustado",
                    "sum",
                ),
            }
        )
    )

    d = d.merge(
        g,
        left_on="idContrato",
        right_index=True,
        how="left",
    )

    d[
        [f"n_{nome}", f"valor_{nome}"]
    ] = d[
        [f"n_{nome}", f"valor_{nome}"]
    ].fillna(0)


n_orig = d["n_orig"].replace(0, np.nan)
valor_orig = d["valor_orig"].replace(0, np.nan)

d["n_parcelas_orig"] = d["n_orig"]
d["prazo_por_parcela"] = d["prazo_total"] / n_orig

d["prop_parc_aditivo"] = (
    d["n_aditivo"] / n_orig
)

d["prop_valor_aditivo"] = (
    d["valor_aditivo"] / valor_orig
)

d["prop_parc_supressao"] = (
    d["n_supressao"] / n_orig
)

d["prop_parc_reajuste"] = (
    d["n_reajuste"] / n_orig
)

# ------------------------------------------------------------
# Aditivos
# ------------------------------------------------------------

d = d.merge(
    aditivos_prazo
    .groupby("idContrato")
    .size()
    .rename("n_aditivos_prazo"),
    left_on="idContrato",
    right_index=True,
    how="left",
)

d = d.merge(
    aditivos_valor
    .groupby("idContrato")
    .size()
    .rename("n_aditivos_valor"),
    left_on="idContrato",
    right_index=True,
    how="left",
)

d[
    ["n_aditivos_prazo", "n_aditivos_valor"]
] = d[
    ["n_aditivos_prazo", "n_aditivos_valor"]
].fillna(0)

d["extensao_prazo_rel"] = (
    d["prazo_ajustado"]
    - d["prazo_total"]
) / d["prazo_total"]

# ------------------------------------------------------------
# Liberações
# ------------------------------------------------------------

lib = (
    liberacoes
    .assign(
        com_atraso=(
            liberacoes["Atraso_considerado"] > 0
        )
    )
    .groupby("id_Contrato")
    .agg(
        pct_lib_com_atraso=(
            "com_atraso",
            "mean",
        ),
        atraso_soma=(
            "Atraso_considerado",
            "sum",
        ),
        abono_soma=(
            "Abono_atraso",
            "sum",
        ),
    )
)

d = d.merge(
    lib,
    left_on="idContrato",
    right_index=True,
    how="left",
)

d["atraso_considerado_soma_rel"] = (
    d["atraso_soma"]
    / d["prazo_total"]
)

d["abono_atraso_soma_rel"] = (
    d["abono_soma"]
    / d["prazo_total"]
)

assert len(d) == len(contratos)
assert d["idContrato"].is_unique

In [5]:
CANDIDATAS_OS = [
    "prazo_total",
    "log_valor_total",
    "perc_desconto",
    "dias_assinatura_ate_os",
    "n_parcelas_orig",
    "prazo_por_parcela",
    "log_valor_por_dia",
    "fornecedor_contratos_anteriores",
    "predio_contratos_anteriores",
]

CANDIDATAS_EXEC = [
    "n_aditivos_prazo",
    "extensao_prazo_rel",
    "n_aditivos_valor",
    "prop_parc_aditivo",
    "prop_valor_aditivo",
    "prop_parc_supressao",
    "prop_parc_reajuste",
    "pct_lib_com_atraso",
    "atraso_considerado_soma_rel",
    "abono_atraso_soma_rel",
]

CATEGORICAS = [
    "Classificacao_contrato",
]

display(
    pd.DataFrame({
        "feature": CANDIDATAS_OS + CANDIDATAS_EXEC,
        "faltantes": d[CANDIDATAS_OS + CANDIDATAS_EXEC].isna().sum().values,
    })
)

,feature,faltantes
0,prazo_total,0
1,log_valor_total,0
2,perc_desconto,0
3,dias_assinatura_ate_os,0
4,n_parcelas_orig,0
5,prazo_por_parcela,1
6,log_valor_por_dia,0
7,fornecedor_contratos_anteriores,0
8,predio_contratos_anteriores,0
9,n_aditivos_prazo,0


## 4. Desenvolvimento, teste final e folds temporais

In [6]:
ctr = (
    d
    .sort_values(
        [
            "data_recebimento_provisorio",
            "idContrato",
        ]
    )
    .reset_index(drop=True)
)

dev = ctr.iloc[:-N_TEST].copy()
test = ctr.iloc[-N_TEST:].copy()

assert (
    dev["data_recebimento_provisorio"].max()
    <= test["data_recebimento_provisorio"].min()
)

folds = []

for k in range(N_FOLDS):

    n_train = MIN_TRAIN + k * BLOCK

    folds.append({
        "fold": k + 1,
        "train": dev.iloc[:n_train].copy(),
        "val": dev.iloc[
            n_train:n_train + BLOCK
        ].copy(),
    })


print("Desenvolvimento:", len(dev))
print("Teste final:", len(test))
print("Contratos de teste:", test["idContrato"].tolist())

display(
    pd.DataFrame([
        {
            "fold": f["fold"],
            "treino": len(f["train"]),
            "validacao": len(f["val"]),
            "atrasados_treino": int(
                f["train"]["target_atraso"].sum()
            ),
            "atrasados_validacao": int(
                f["val"]["target_atraso"].sum()
            ),
        }
        for f in folds
    ])
)

Desenvolvimento: 40
Teste final: 8
Contratos de teste: [173, 201, 287, 283, 342, 317, 254, 377]


,fold,treino,validacao,atrasados_treino,atrasados_validacao
0,1,20,5,8,3
1,2,25,5,11,2
2,3,30,5,13,0
3,4,35,5,13,2


## 5. Remoção de redundância

A poda usa somente os contratos de desenvolvimento e **não usa o target**.

In [7]:
def podar_redundantes(
    frame,
    candidatas,
    limite=LIMITE_CORRELACAO,
):

    corr = frame[candidatas].corr().abs()

    mantidas = []
    descartadas = []

    for col in candidatas:

        if frame[col].nunique(dropna=True) <= 1:
            descartadas.append(
                (
                    col,
                    "constante",
                    None,
                    None,
                )
            )
            continue

        parceiros = [
            m
            for m in mantidas
            if corr.loc[col, m] > limite
        ]

        if parceiros:
            melhor = max(
                parceiros,
                key=lambda m: corr.loc[col, m],
            )

            descartadas.append(
                (
                    col,
                    "redundante",
                    melhor,
                    corr.loc[col, melhor],
                )
            )

        else:
            mantidas.append(col)

    return (
        mantidas,
        pd.DataFrame(
            descartadas,
            columns=[
                "feature",
                "motivo",
                "parecida_com",
                "correlacao",
            ],
        ),
    )


CENARIOS = {}

for nome, candidatas in {
    "na ordem de serviço": CANDIDATAS_OS,
    "com dados da execução": (
        CANDIDATAS_OS
        + CANDIDATAS_EXEC
    ),
}.items():

    mantidas, descartadas = (
        podar_redundantes(
            dev,
            candidatas,
        )
    )

    CENARIOS[nome] = mantidas

    print(
        f"{nome}: "
        f"{len(candidatas)} candidatas "
        f"-> {len(mantidas)} mantidas"
    )

    print("Mantidas:", mantidas)

    if len(descartadas):
        display(descartadas)

na ordem de serviço: 9 candidatas -> 7 mantidas
Mantidas: ['prazo_total', 'log_valor_total', 'perc_desconto', 'dias_assinatura_ate_os', 'prazo_por_parcela', 'fornecedor_contratos_anteriores', 'predio_contratos_anteriores']


,feature,motivo,parecida_com,correlacao
0,n_parcelas_orig,redundante,prazo_total,0.927
1,log_valor_por_dia,redundante,log_valor_total,0.894


com dados da execução: 19 candidatas -> 17 mantidas
Mantidas: ['prazo_total', 'log_valor_total', 'perc_desconto', 'dias_assinatura_ate_os', 'prazo_por_parcela', 'fornecedor_contratos_anteriores', 'predio_contratos_anteriores', 'n_aditivos_prazo', 'extensao_prazo_rel', 'n_aditivos_valor', 'prop_parc_aditivo', 'prop_valor_aditivo', 'prop_parc_supressao', 'prop_parc_reajuste', 'pct_lib_com_atraso', 'atraso_considerado_soma_rel', 'abono_atraso_soma_rel']


,feature,motivo,parecida_com,correlacao
0,n_parcelas_orig,redundante,prazo_total,0.927
1,log_valor_por_dia,redundante,log_valor_total,0.894


## 6. Pré-processamento

A mesma transformação é usada nas duas etapas:

- mediana + padronização nas variáveis numéricas;
- one-hot encoding na variável categórica.

Tudo é ajustado somente no conjunto de treino de cada fold.

In [8]:
def make_prep(num, cat):

    return ColumnTransformer([
        (
            "num",
            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                (
                    "sc",
                    StandardScaler(),
                ),
            ]),
            num,
        ),

        (
            "cat",
            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="most_frequent",
                    ),
                ),
                (
                    "ohe",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            cat,
        ),
    ])

## 7. Modelos das duas etapas

Para evitar excesso de ajuste em apenas 48 contratos, a grade é pequena.

**Etapa 1:** regressão logística regularizada ou Random Forest raso.

**Etapa 2:** Ridge com `log1p` no atraso relativo, treinado **somente nos contratos que realmente atrasaram**.

In [9]:
def classificadores():

    candidatos = []

    for c in [0.1, 1.0, 10.0]:

        candidatos.append((
            f"Logistic C={c}",
            LogisticRegression(
                C=c,
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ))

    candidatos.append((
        "RandomForest raso",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=3,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ))

    return candidatos


def regressores_severidade():

    return [
        (
            f"Ridge log1p alpha={alpha}",
            TransformedTargetRegressor(
                regressor=Ridge(
                    alpha=alpha,
                ),
                func=np.log1p,
                inverse_func=np.expm1,
            ),
        )
        for alpha in [10, 100, 1000]
    ]

## 8. Funções de ajuste e métricas

A previsão final em dias é uma **esperança matemática**:

`P(atraso) × severidade prevista se atrasar × prazo ajustado`

Assim, não precisamos escolher threshold para converter risco em dias.

In [10]:
def safe_auc(y_true, prob):

    if pd.Series(y_true).nunique() < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob,
    )


def safe_pr_auc(y_true, prob):

    if pd.Series(y_true).nunique() < 2:
        return np.nan

    return average_precision_score(
        y_true,
        prob,
    )


def metricas_duas_etapas(frame):

    y_dias = frame["target_dias_atraso"]
    pred_dias = frame["pred_dias"]

    return {
        "mae_dias": mean_absolute_error(
            y_dias,
            pred_dias,
        ),
        "rmse_dias": np.sqrt(
            mean_squared_error(
                y_dias,
                pred_dias,
            )
        ),
        "erro_assinado_medio": (
            pred_dias - y_dias
        ).mean(),
        "spearman_dias": (
            spearmanr(
                y_dias,
                pred_dias,
            )[0]
            if pred_dias.nunique() > 1
            else np.nan
        ),
        "auc_atrasa": safe_auc(
            frame["target_atraso"],
            frame["prob_atraso"],
        ),
        "pr_auc_atrasa": safe_pr_auc(
            frame["target_atraso"],
            frame["prob_atraso"],
        ),
        "brier": brier_score_loss(
            frame["target_atraso"],
            frame["prob_atraso"],
        ),
    }


def ajustar_duas_etapas(
    treino,
    alvo,
    num,
    cat,
    clf_estimator,
    sev_estimator,
):

    features = num + cat

    # --------------------------------------------------------
    # ETAPA 1 — probabilidade de atraso
    # --------------------------------------------------------

    clf = Pipeline([
        (
            "prep",
            make_prep(
                num,
                cat,
            ),
        ),
        (
            "model",
            clf_estimator,
        ),
    ])

    clf.fit(
        treino[features],
        treino["target_atraso"],
    )

    prob_atraso = clf.predict_proba(
        alvo[features]
    )[:, 1]

    # --------------------------------------------------------
    # ETAPA 2 — severidade somente entre atrasados
    # --------------------------------------------------------

    treino_pos = treino[
        treino["target_atraso"].eq(1)
    ].copy()

    if len(treino_pos) < 4:
        raise ValueError(
            "Poucos contratos atrasados para a etapa 2."
        )

    sev = Pipeline([
        (
            "prep",
            make_prep(
                num,
                cat,
            ),
        ),
        (
            "model",
            sev_estimator,
        ),
    ])

    sev.fit(
        treino_pos[features],
        treino_pos["target_atraso_rel"],
    )

    severidade_rel = np.maximum(
        0,
        sev.predict(
            alvo[features]
        ),
    )

    # --------------------------------------------------------
    # PREVISÃO FINAL
    # --------------------------------------------------------

    pred_rel_esperado = (
        prob_atraso
        * severidade_rel
    )

    pred_dias = (
        pred_rel_esperado
        * alvo["prazo_ajustado"].to_numpy()
    )

    saida = alvo.copy()

    saida["prob_atraso"] = prob_atraso
    saida["severidade_rel_se_atrasar"] = severidade_rel
    saida["pred_rel_esperado"] = pred_rel_esperado
    saida["pred_dias"] = pred_dias

    return clf, sev, saida

## 9. Baselines

Dois baselines ajudam a interpretar o ganho real:

- **zero:** sempre prevê 0 dias;
- **hurdle constante:** prevalência de atraso × severidade média dos atrasados.

In [11]:
def prever_baselines(treino, alvo):

    out = alvo.copy()

    # Baseline 1: sempre zero.
    out["baseline_zero"] = 0.0

    # Baseline 2: duas etapas constantes.
    p = treino["target_atraso"].mean()

    positivos = treino.loc[
        treino["target_atraso"].eq(1),
        "target_atraso_rel",
    ]

    sev = positivos.mean()

    out["baseline_hurdle"] = (
        p
        * sev
        * out["prazo_ajustado"]
    )

    return out


def metricas_baseline(frame, col):

    pred = frame[col]
    real = frame["target_dias_atraso"]

    return {
        "mae_dias": mean_absolute_error(
            real,
            pred,
        ),
        "rmse_dias": np.sqrt(
            mean_squared_error(
                real,
                pred,
            )
        ),
        "erro_assinado_medio": (
            pred - real
        ).mean(),
    }

## 10. Validação temporal

Para cada cenário, combinamos os classificadores com os regressores de severidade.

O critério principal de seleção é **RMSE em dias agregado nas validações temporais**, porque o MAE tende a favorecer excessivamente a previsão zero nesta base.

In [12]:
resultados = {}

for cenario, num in CENARIOS.items():

    ranking = []
    guardados = {}

    for clf_nome, clf_est in classificadores():

        for sev_nome, sev_est in regressores_severidade():

            oof_parts = []

            for fold in folds:

                _, _, pred = ajustar_duas_etapas(
                    treino=fold["train"],
                    alvo=fold["val"],
                    num=num,
                    cat=CATEGORICAS,
                    clf_estimator=clf_est,
                    sev_estimator=sev_est,
                )

                pred["fold"] = fold["fold"]

                oof_parts.append(pred)

            oof = pd.concat(
                oof_parts,
                ignore_index=True,
            )

            m = metricas_duas_etapas(oof)

            nome = (
                f"{clf_nome} + {sev_nome}"
            )

            ranking.append({
                "modelo": nome,
                **m,
            })

            guardados[nome] = {
                "clf": clf_est,
                "sev": sev_est,
                "oof": oof,
            }

    ranking = (
        pd.DataFrame(ranking)
        .sort_values(
            [
                "rmse_dias",
                "mae_dias",
            ]
        )
        .reset_index(drop=True)
    )

    melhor_nome = ranking.iloc[0]["modelo"]

    resultados[cenario] = {
        "ranking": ranking,
        "melhor_nome": melhor_nome,
        **guardados[melhor_nome],
    }

    print("\n==", cenario, "==")
    display(ranking.head(10))


== na ordem de serviço ==


,modelo,mae_dias,rmse_dias,erro_assinado_medio,spearman_dias,auc_atrasa,pr_auc_atrasa,brier
0,Logistic C=0.1 + Ridge log1p alpha=100,43.364,59.452,25.053,0.289,0.341,0.315,0.283
1,Logistic C=1.0 + Ridge log1p alpha=100,43.364,59.959,15.584,0.104,0.352,0.307,0.375
2,Logistic C=1.0 + Ridge log1p alpha=1000,43.606,60.257,26.977,0.182,0.352,0.307,0.375
3,RandomForest raso + Ridge log1p alpha=100,45.964,65.252,23.207,0.196,0.242,0.270,0.287
4,Logistic C=0.1 + Ridge log1p alpha=1000,51.149,71.501,41.671,0.402,0.341,0.315,0.283
5,Logistic C=10.0 + Ridge log1p alpha=100,50.447,75.466,7.513,-0.168,0.374,0.311,0.467
6,Logistic C=10.0 + Ridge log1p alpha=1000,51.783,75.730,12.294,-0.049,0.374,0.311,0.467
7,Logistic C=0.1 + Ridge log1p alpha=10,48.544,77.258,-0.805,-0.330,0.341,0.315,0.283
8,Logistic C=1.0 + Ridge log1p alpha=10,46.395,79.502,-2.562,-0.307,0.352,0.307,0.375
9,RandomForest raso + Ridge log1p alpha=10,51.299,80.812,0.914,-0.410,0.242,0.270,0.287



== com dados da execução ==


,modelo,mae_dias,rmse_dias,erro_assinado_medio,spearman_dias,auc_atrasa,pr_auc_atrasa,brier
0,RandomForest raso + Ridge log1p alpha=100,41.762,57.659,16.927,0.386,0.495,0.400,0.256
1,Logistic C=0.1 + Ridge log1p alpha=100,41.855,59.612,16.537,0.438,0.582,0.467,0.253
2,Logistic C=1.0 + Ridge log1p alpha=100,47.605,70.901,13.171,0.060,0.429,0.328,0.348
3,Logistic C=0.1 + Ridge log1p alpha=1000,49.317,74.792,41.911,0.462,0.582,0.467,0.253
4,RandomForest raso + Ridge log1p alpha=10,42.583,76.634,-4.860,-0.160,0.495,0.400,0.256
5,RandomForest raso + Ridge log1p alpha=1000,51.539,80.328,44.772,0.383,0.495,0.400,0.256
6,Logistic C=1.0 + Ridge log1p alpha=1000,55.798,83.543,33.953,0.180,0.429,0.328,0.348
7,Logistic C=0.1 + Ridge log1p alpha=10,47.688,85.080,-0.142,-0.120,0.582,0.467,0.253
8,Logistic C=10.0 + Ridge log1p alpha=100,56.316,86.199,12.612,-0.187,0.374,0.309,0.525
9,Logistic C=1.0 + Ridge log1p alpha=10,51.886,89.970,3.481,-0.120,0.429,0.328,0.348


## 11. Baselines na validação temporal

In [13]:
baseline_rows = []

for fold in folds:

    b = prever_baselines(
        fold["train"],
        fold["val"],
    )

    for col in [
        "baseline_zero",
        "baseline_hurdle",
    ]:

        baseline_rows.append({
            "fold": fold["fold"],
            "baseline": col,
            **metricas_baseline(
                b,
                col,
            ),
        })

baseline_cv = pd.DataFrame(
    baseline_rows
)

display(
    baseline_cv
    .groupby(
        "baseline",
        as_index=False,
    )
    .agg(
        mae_medio=("mae_dias", "mean"),
        rmse_medio=("rmse_dias", "mean"),
        erro_assinado_medio=(
            "erro_assinado_medio",
            "mean",
        ),
    )
)

,baseline,mae_medio,rmse_medio,erro_assinado_medio
0,baseline_hurdle,82.923,106.371,82.923
1,baseline_zero,27.900,52.053,-27.900


## 12. Teste final reservado

Somente depois da escolha por validação temporal, o melhor par de modelos é ajustado nos 40 contratos de desenvolvimento e avaliado nos 8 mais recentes.

In [14]:
teste_rows = []
tabelas_teste = {}

for cenario, num in CENARIOS.items():

    r = resultados[cenario]

    clf_final, sev_final, pred = (
        ajustar_duas_etapas(
            treino=dev,
            alvo=test,
            num=num,
            cat=CATEGORICAS,
            clf_estimator=r["clf"],
            sev_estimator=r["sev"],
        )
    )

    metricas = metricas_duas_etapas(
        pred
    )

    teste_rows.append({
        "cenario": cenario,
        "modelo": r["melhor_nome"],
        **metricas,
    })

    tabelas_teste[cenario] = pred

teste_resultados = pd.DataFrame(
    teste_rows
)

display(teste_resultados)

,cenario,modelo,mae_dias,rmse_dias,erro_assinado_medio,spearman_dias,auc_atrasa,pr_auc_atrasa,brier
0,na ordem de serviço,Logistic C=0.1 + Ridge log1p alpha=100,72.886,82.618,41.060,-0.514,0.167,0.225,0.270
1,com dados da execução,RandomForest raso + Ridge log1p alpha=100,68.677,87.039,39.519,-0.171,0.833,0.583,0.230


## 13. Comparação com os baselines no teste

In [15]:
baseline_test = prever_baselines(
    dev,
    test,
)

baseline_test_rows = []

for col in [
    "baseline_zero",
    "baseline_hurdle",
]:

    baseline_test_rows.append({
        "modelo": col,
        **metricas_baseline(
            baseline_test,
            col,
        ),
    })

display(
    pd.DataFrame(
        baseline_test_rows
    )
)

,modelo,mae_dias,rmse_dias,erro_assinado_medio
0,baseline_zero,21.875,54.352,-21.875
1,baseline_hurdle,139.288,161.907,109.955


## 14. Previsões contrato a contrato

Esta tabela é importante para verificar se o modelo está:

- superestimando contratos sem atraso;
- subestimando atrasos grandes;
- produzindo probabilidades coerentes;
- comprimindo excessivamente a severidade.

In [16]:
for cenario, t in tabelas_teste.items():

    print("\n==", cenario, "==")

    detalhe = t[
        [
            "idContrato",
            "prazo_ajustado",
            "target_atraso",
            "target_dias_atraso",
            "prob_atraso",
            "severidade_rel_se_atrasar",
            "pred_dias",
        ]
    ].copy()

    detalhe["erro_dias"] = (
        detalhe["pred_dias"]
        - detalhe["target_dias_atraso"]
    )

    display(
        detalhe.round(3)
    )


== na ordem de serviço ==


,idContrato,prazo_ajustado,target_atraso,target_dias_atraso,prob_atraso,severidade_rel_se_atrasar,pred_dias,erro_dias
40,173,1110,0,0,0.572,0.170,108.077,108.077
41,201,881,0,0,0.527,0.219,101.477,101.477
42,287,360,0,0,0.460,0.299,49.459,49.459
43,283,421,0,0,0.501,0.363,76.688,76.688
44,342,60,0,0,0.381,0.551,12.607,12.607
45,317,390,1,23,0.338,0.333,43.869,20.869
46,254,750,0,0,0.503,0.230,86.608,86.608
47,377,135,1,152,0.493,0.371,24.699,-127.301



== com dados da execução ==


,idContrato,prazo_ajustado,target_atraso,target_dias_atraso,prob_atraso,severidade_rel_se_atrasar,pred_dias,erro_dias
40,173,1110,0,0,0.494,0.103,56.707,56.707
41,201,881,0,0,0.707,0.311,193.524,193.524
42,287,360,0,0,0.463,0.297,49.409,49.409
43,283,421,0,0,0.500,0.260,54.750,54.750
44,342,60,0,0,0.270,0.441,7.139,7.139
45,317,390,1,23,0.609,0.002,0.525,-22.475
46,254,750,0,0,0.522,0.182,71.256,71.256
47,377,135,1,152,0.637,0.673,57.840,-94.160


## 15. Diagnóstico separado das duas etapas

Uma previsão final ruim pode vir de duas fontes diferentes:

- a classificação não identifica corretamente quem atrasa;
- a classificação funciona, mas a regressão de severidade erra o tamanho do atraso.

A tabela abaixo mostra o erro de severidade **somente nos contratos atrasados do teste**.

In [17]:
for cenario, t in tabelas_teste.items():

    positivos = t[
        t["target_atraso"].eq(1)
    ].copy()

    if len(positivos) == 0:
        continue

    positivos["severidade_real_rel"] = (
        positivos["target_atraso_rel"]
    )

    positivos["erro_severidade_rel"] = (
        positivos[
            "severidade_rel_se_atrasar"
        ]
        - positivos[
            "severidade_real_rel"
        ]
    )

    print("\n==", cenario, "==")

    display(
        positivos[
            [
                "idContrato",
                "target_dias_atraso",
                "prob_atraso",
                "severidade_real_rel",
                "severidade_rel_se_atrasar",
                "erro_severidade_rel",
            ]
        ].round(3)
    )


== na ordem de serviço ==


,idContrato,target_dias_atraso,prob_atraso,severidade_real_rel,severidade_rel_se_atrasar,erro_severidade_rel
45,317,23,0.338,0.059,0.333,0.274
47,377,152,0.493,1.126,0.371,-0.755



== com dados da execução ==


,idContrato,target_dias_atraso,prob_atraso,severidade_real_rel,severidade_rel_se_atrasar,erro_severidade_rel
45,317,23,0.609,0.059,0.002,-0.057
47,377,152,0.637,1.126,0.673,-0.453


## 16. Exportação

In [18]:
OUTPUT_DIR = Path(
    "10_modelo_duas_etapas_outputs"
)

OUTPUT_DIR.mkdir(
    exist_ok=True,
)

for cenario, r in resultados.items():

    nome = (
        cenario
        .replace(" ", "_")
        .replace("ç", "c")
        .replace("ã", "a")
    )

    r["ranking"].to_csv(
        OUTPUT_DIR
        / f"ranking_{nome}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    r["oof"].to_csv(
        OUTPUT_DIR
        / f"oof_{nome}.csv",
        index=False,
        encoding="utf-8-sig",
    )

for cenario, t in tabelas_teste.items():

    nome = (
        cenario
        .replace(" ", "_")
        .replace("ç", "c")
        .replace("ã", "a")
    )

    t.to_csv(
        OUTPUT_DIR
        / f"teste_{nome}.csv",
        index=False,
        encoding="utf-8-sig",
    )

teste_resultados.to_csv(
    OUTPUT_DIR / "resumo_teste_final.csv",
    index=False,
    encoding="utf-8-sig",
)

baseline_cv.to_csv(
    OUTPUT_DIR / "baselines_cv.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Resultados salvos em:",
    OUTPUT_DIR.resolve(),
)

Resultados salvos em: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\notebooks\10_modelo_duas_etapas_outputs


## Como interpretar o resultado

O modelo em duas etapas só representa avanço real se:

- superar o baseline zero em **RMSE** sem piorar excessivamente o **MAE**;
- apresentar AUC/PR-AUC razoáveis e relativamente estáveis na etapa 1;
- evitar previsões altas para muitos contratos sem atraso;
- melhorar a severidade dos contratos realmente atrasados;
- manter desempenho semelhante entre validação temporal e teste final.

Com apenas 48 contratos e poucos atrasos positivos, a etapa 2 continua sendo estatisticamente pequena. Por isso, este notebook usa modelos simples e forte regularização.